In [18]:
import csv, os, re, time, datetime as dt
import requests
from bs4 import BeautifulSoup

##BASICS
EMAIL = "B32895@student.ucu.ac.ug" ##identifying school email
CATEGORIES = ["/phones-tablets/", "/baby-products/", "/health-beauty/"]##As agreed on by group one.
KEYWORDS = {"/baby-products/": "baby", "/health-beauty/": "health"}

In [19]:
Identifier = f"DSC2107-Course-Project ({EMAIL})"
BASE = "https://www.jumia.com.eg"
HEADERS = {"User-Agent": Identifier}
DELAY = 3 #seconds between requests
OUT = "recon_output" ##name of the database we save the data in
os.makedirs(OUT, exist_ok=True)
n_requests = 0

In [20]:
def polite_get(url, retries=3):
    """GET with fixed delay, identifying User-Agent, retry + back-off."""
    global n_requests
    for attempt in range(1, retries + 1):
        time.sleep(DELAY)
        n_requests += 1
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code == 200:
                return r
            print(f"  ! {url} -> HTTP {r.status_code} (attempt {attempt})")
            if r.status_code in (403, 429):   # blocked: stop, document, slow down
                time.sleep(30 * attempt)
        except requests.RequestException as e:
            print(f"  ! {url} -> {e} (attempt {attempt})")
            time.sleep(5 * attempt)
    return None

In [21]:
##robots.txt
#NOTE: Python's urllib.robotparser does NOT understand '*' wildcards inside paths e.g. Disallow: /*price=, so it would wrongly say "allowed".
#This small matcher follows the standard longest-match rule instead.

def star_rules(txt):
    groups, cur, last_ua = [], None, False
    for raw in txt.splitlines():
        line = raw.split("#")[0].strip()
        if ":" not in line:
            continue
        k, v = [x.strip() for x in line.split(":", 1)]
        k = k.lower()
        if k == "user-agent":
            if not last_ua:
                cur = {"agents": [], "rules": []}
                groups.append(cur)
            cur["agents"].append(v.lower())
            last_ua = True
        elif k in ("allow", "disallow") and cur is not None:
            cur["rules"].append((k, v))
            last_ua = False
    return [r for g in groups if "*" in g["agents"] for r in g["rules"]]

In [22]:
def rule_regex(pat):
    anchored = pat.endswith("$")
    pat = pat.rstrip("$")
    rx = "".join(".*" if c == "*" else re.escape(c) for c in pat)
    return re.compile("^" + rx + ("$" if anchored else ""))

In [23]:
def check(rules, path):
    best_len, ok, why = -1, True, "no matching rule (allowed by default)"
    for kind, pat in rules:
        if pat and rule_regex(pat).match(path):
            if len(pat) > best_len or (len(pat) == best_len and kind == "allow"):
                best_len, ok, why = len(pat), kind == "allow", f"{kind.capitalize()}: {pat}"
    return ok, why

In [24]:
def main():
    started = dt.datetime.now().isoformat(timespec="seconds")
    report = [f"# Jumia Egypt reconnaissance output\nRun at {started}\nUser-Agent: {Identifier}\n"]

    # 1. robots.txt
    print("1. robots.txt")
    r = polite_get(f"{BASE}/robots.txt")
    if r is None:
        print("Could not fetch robots.txt. Stop and document this.")
        return
    open(f"{OUT}/robots_jumia_eg.txt", "w", encoding="utf-8").write(r.text)
    rules = star_rules(r.text)
    report.append(f"## robots.txt\n- {len(rules)} rules in the 'User-agent: *' group\n"
                  "- Header rules: identify the bot, owner contact required, under 200 requests/minute, "
                  "no fake user-agents, no IP rotation\n")

    #URLs we plan to use AND URLs we must avoid (they are tested, never fetched)
    tests = [(c, "planned category") for c in CATEGORIES]
    tests += [(CATEGORIES[0] + "?page=2", "planned pagination"),
              (CATEGORIES[0] + "?sort=price", "must AVOID: sort facet"),
              (CATEGORIES[0] + "?price=1000-5000", "must AVOID: price facet"),
              ("/en" + CATEGORIES[0], "must AVOID: /en/ folder"),
              ("/mobapi/", "must AVOID: mobile API"),
              (CATEGORIES[0] + "?q=phone", "must AVOID: site search"),
              ("/brands/", "must AVOID: brand pages"),
              (CATEGORIES[0] + "top-sellers/", "must AVOID: top-sellers"),
              ("/all-products/", "must AVOID: all-products"),
              ("/ar" + CATEGORIES[0], "Arabic folder (not used)")]
    rows = []
    report.append("| Path | Purpose | Allowed | Matching rule |\n|---|")
    for path, purpose in tests:
        ok, why = check(rules, path)
        rows.append([path, purpose, ok, why])
        report.append(f"| `{path}` | {purpose} | {ok} | {why} |")
        print(f"  {path:40s} {ok}   [{why}]")
    with open(f"{OUT}/robots_checks.csv", "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows([["path", "purpose", "allowed", "rule"]] + rows)

    # 2. Category pages: HTML vs JS, card fields ------------------------------
    report.append("\n## Category pages (page 1)\n| Category | HTTP | Cards on page | Pages in pagination |\n|---|---|---|---|")
    first_soup = None

    for cat in CATEGORIES:
            if not check(rules, cat)[0]:
                report.append(f"| {cat} | skipped (disallowed) | - | - |")
                continue
            print(f"2. {cat}")
            r = polite_get(BASE + cat, retries=1)
            if r is None and cat in KEYWORDS:
                # slug may differ: look for it in the homepage menu (one extra request)
                home = polite_get(BASE + "/", retries=1)
                if home:
                    hs = BeautifulSoup(home.text, "html.parser")
                    found = sorted({a["href"] for a in hs.select("a[href^='/']")
                                    if KEYWORDS[cat] in a["href"].lower()
                                    and a["href"].count("/") <= 2 and "--" not in a["href"]})
                    print(f"   {cat} not found. Homepage links containing "
                          f"'{KEYWORDS[cat]}': {found}")
                    report.append(f"\n*{cat} failed; candidate slugs from homepage: {found}*\n")
            if r is None:
                report.append(f"| {cat} | failed | - | - |")
                continue
            soup = BeautifulSoup(r.text, "html.parser")
            cards = soup.select("article.prd")
            pages = [int(m) for a in soup.select("a[href*='page=']")
                     for m in re.findall(r"page=(\d+)", a["href"])]
            slug = cat.strip("/").replace("/", "_")
            open(f"{OUT}/page1_{slug}.html", "w", encoding="utf-8").write(r.text)
            report.append(f"| {cat} | 200 | {len(cards)} | {max(pages) if pages else 'not found'} |")
            print(f"   cards: {len(cards)}  max page link: {max(pages) if pages else None}")
            if first_soup is None and cards:
                first_soup = soup

            # 3. Field analysis on the first category --------------------------------
                if first_soup is not None:
                    cards = first_soup.select("article.prd")
                    open(f"{OUT}/first_card.html", "w", encoding="utf-8").write(cards[0].prettify())
                    core = cards[0].select_one("a.core")
                    data_attrs = sorted(k for k in (core.attrs if core else {}) if k.startswith("data-"))
                    probes = {  # candidate selectors, adjust if a count is 0
                        "name": "h3.name", "price": "div.prc", "old price": "div.old",
                        "discount badge": "div.bdg._dsct", "rating": "div.stars._s",
                        "review count": "div.rev", "link": "a.core"}
                    report.append("\n## Field availability on page 1 of the first category\n| Field | Selector | Cards with it |\n|---|---|---|")
                    for name, sel in probes.items():
                        have = sum(1 for c in cards if c.select_one(sel))
                        report.append(f"| {name} | `{sel}` | {have}/{len(cards)} |")
                    report.append(f"\nData attributes on the main link (usable for brand/category/id): `{data_attrs}`")
                    has_ld = len(first_soup.select('script[type="application/ld+json"]'))
                    report.append(f"\nJSON-LD blocks in page: {has_ld}")
                    print("3. data- attributes:", data_attrs)

                    # 4. Pagination check: page 2 must differ from page 1
                    cat0 = CATEGORIES[0]
                    if check(rules, cat0 + "?page=2")[0]:
                                print("4. pagination test")
                                r2 = polite_get(f"{BASE}{cat0}?page=2")
                                if r2:
                                    s2 = BeautifulSoup(r2.text, "html.parser")
                                    links1 = {a["href"] for a in first_soup.select("article.prd a.core")}
                                    links2 = {a["href"] for a in s2.select("article.prd a.core")}
                                    report.append(f"\n## Pagination\n- `?page=2` returned {len(links2)} cards; "
                                                  f"overlap with page 1: {len(links1 & links2)} (expect 0)")
                    else:
                            report.append("\n**No `article.prd` cards found: the page may be JavaScript-rendered. "
                                          "Check the Network tab for a JSON call (NOT /mobapi/) or use Playwright.**")
                    
                    report.append(f"\nTotal requests made by this script: {n_requests}")
                    open(f"{OUT}/recon_report.md", "w", encoding="utf-8").write("\n".join(report))
                    print(f"\nDone. {n_requests} requests. See {OUT}/recon_report.md")

In [25]:

if __name__ == "__main__":
    main()

1. robots.txt
  /phones-tablets/                         True   [no matching rule (allowed by default)]
  /baby-products/                          True   [no matching rule (allowed by default)]
  /health-beauty/                          True   [no matching rule (allowed by default)]
  /phones-tablets/?page=2                  True   [no matching rule (allowed by default)]
  /phones-tablets/?sort=price              False   [Disallow: /*sort=]
  /phones-tablets/?price=1000-5000         False   [Disallow: /*price=]
  /en/phones-tablets/                      False   [Disallow: /en/]
  /mobapi/                                 False   [Disallow: /mobapi/]
  /phones-tablets/?q=phone                 False   [Disallow: /*q=]
  /brands/                                 False   [Disallow: */brands/]
  /phones-tablets/top-sellers/             False   [Disallow: */top-sellers/]
  /all-products/                           False   [Disallow: */all-products/]
  /ar/phones-tablets/                      Tr